<a href="https://colab.research.google.com/github/AkiraTheSquid/ARENA_3.0/blob/main/chapter0_fundamentals/exercises/part1_ray_tracing/0.1_Ray_Tracing_supplementary.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [0.1] - Ray Tracing (supplementary exercises for `make_rays_1d`)

This notebook is a companion to the **first exercise** of [0.1] Ray Tracing, `make_rays_1d`. That exercise asks you to build a tensor of shape `(num_pixels, 2, 3)` where each row is `(origin, direction)` and the directions fan out from `(1, -y_limit, 0)` to `(1, +y_limit, 0)`.

Solving it cleanly needs four prerequisite skills. Each skill gets its own section below, and every section has the same structure:

1. **Worked examples** — read the code, run it, predict the output *before* running.
2. **Faded problems** — most of the code is given; fill in the `...` blanks.
3. **Challenge problems** — write the whole thing yourself. These probe the edges of the skill.

Section 5 then gives you several *sibling* problems at the same difficulty as `make_rays_1d` itself.

Every problem comes with `assert`-based tests in the same cell — a cell that runs with no error is a pass. Solutions are hidden in dropdowns underneath.

| Section | Prerequisite |
|---|---|
| 1️⃣ | Creating tensors of a given shape & dtype (`zeros`, `ones`, `full`, `tensor`) |
| 2️⃣ | Evenly-spaced values: `linspace` vs `arange` (endpoints, counts, steps) |
| 3️⃣ | Writing into slices of a tensor (views, broadcasting on assignment, `out=`) |
| 4️⃣ | Rays as `(origin, direction)` pairs — the geometry behind the tensor layout |
| 5️⃣ | Problems at the level of `make_rays_1d` |

## Setup

In [ ]:
import torch as t
from torch import Tensor

def check(got, expected, name="test", atol=1e-6):
    """Tiny test helper: compares shape, dtype and values."""
    expected = t.as_tensor(expected, dtype=got.dtype) if not isinstance(expected, Tensor) else expected
    assert got.shape == expected.shape, f"{name}: shape {tuple(got.shape)} != {tuple(expected.shape)}"
    assert got.dtype == expected.dtype, f"{name}: dtype {got.dtype} != {expected.dtype}"
    assert t.allclose(got, expected, atol=atol), f"{name}: values differ\n got={got}\n exp={expected}"
    print(f"{name}: ok")

# 1️⃣ Creating tensors of a given shape & dtype

`make_rays_1d` starts with an empty container: `t.zeros((num_pixels, 2, 3), dtype=t.float32)`. You need to be fluent in describing a shape as a tuple, reading it back off a tensor, and controlling the dtype.

## Worked examples

For each cell, write down what you expect `shape` and `dtype` to be, then run it.

In [ ]:
# Example 1.1 — shape is a tuple; each entry is the length of one axis
a = t.zeros((4, 2, 3))
print(a.shape, a.dtype)        # torch.Size([4, 2, 3]) torch.float32  (float32 is the default)
print(a[0])                    # the first "row": a (2, 3) block of zeros

In [ ]:
# Example 1.2 — dtype is chosen at creation; ints and floats are different tensors
b = t.ones((3,), dtype=t.int64)
c = t.ones(3)                  # shape can be given as bare ints instead of a tuple
print(b, b.dtype)
print(c, c.dtype)
print((b + c).dtype)           # mixing → promotes to float32

In [ ]:
# Example 1.3 — full() fills with a constant; tensor() builds from nested Python lists
d = t.full((2, 3), 7.0)
e = t.tensor([[0, 0, 0], [1, -1.0, 0]])   # one float in the list → whole tensor is float32
print(d)
print(e, e.shape, e.dtype)

In [ ]:
# Example 1.4 — *_like copies shape AND dtype from another tensor
f = t.zeros_like(e)
print(f.shape, f.dtype)
print(t.empty((2, 2)))         # empty() does NOT zero memory: garbage values, fast. Avoid unless you overwrite everything.

## Faded problems

Fill in the `...`. Do not change anything else.

In [ ]:
# Faded 1.1 — a (5, 2, 3) block of zeros, float32
rays = t.zeros(..., dtype=...)
check(rays, t.zeros((5, 2, 3)), "faded 1.1")

<details><summary>Solution</summary>

```python
rays = t.zeros((5, 2, 3), dtype=t.float32)
```
</details>

In [ ]:
# Faded 1.2 — a length-4 integer tensor of all 1s, and a length-4 float tensor of all -2.5
ones_int = t.ones(..., dtype=...)
neg = t.full(..., ...)
assert ones_int.dtype == t.int64 and ones_int.tolist() == [1, 1, 1, 1], "faded 1.2 ones_int"
check(neg, t.tensor([-2.5, -2.5, -2.5, -2.5]), "faded 1.2 neg")

<details><summary>Solution</summary>

```python
ones_int = t.ones((4,), dtype=t.int64)
neg = t.full((4,), -2.5)
```
</details>

In [ ]:
# Faded 1.3 — build this exact tensor from a Python list literal:
# [[0., 0., 0.],
#  [1., 0.5, 0.]]
pair = t.tensor(...)
check(pair, t.tensor([[0.0, 0.0, 0.0], [1.0, 0.5, 0.0]]), "faded 1.3")

<details><summary>Solution</summary>

```python
pair = t.tensor([[0.0, 0.0, 0.0], [1.0, 0.5, 0.0]])
```
</details>

In [ ]:
# Faded 1.4 — same shape and dtype as `template`, but filled with 9s. Use a *_like function.
template = t.zeros((3, 2), dtype=t.float64)
nines = t.full_like(..., ...)
check(nines, t.full((3, 2), 9.0, dtype=t.float64), "faded 1.4")

<details><summary>Solution</summary>

```python
nines = t.full_like(template, 9.0)
```
</details>

## Challenge problems

In [ ]:
# Challenge 1.1 — implement `blank_rays(n)`: shape (n, 2, 3), float32, all zeros.
# It must work for n = 0 as well (an empty batch is still a valid tensor).
def blank_rays(n: int) -> Tensor:
    raise NotImplementedError()

check(blank_rays(6), t.zeros((6, 2, 3)), "challenge 1.1 n=6")
check(blank_rays(0), t.zeros((0, 2, 3)), "challenge 1.1 n=0")
assert blank_rays(3).dtype == t.float32

<details><summary>Solution</summary>

```python
def blank_rays(n: int) -> Tensor:
    return t.zeros((n, 2, 3), dtype=t.float32)
```
</details>

In [ ]:
# Challenge 1.2 — predict, then verify. Fill each `expected_*` with the literal you predict.
x = t.tensor([1, 2, 3])
y = t.tensor([1.0, 2, 3])
z = t.tensor([True, False])

expected_x_dtype = ...   # e.g. t.int64
expected_y_dtype = ...
expected_z_dtype = ...
expected_xy_dtype = ...  # dtype of x + y

assert x.dtype == expected_x_dtype, x.dtype
assert y.dtype == expected_y_dtype, y.dtype
assert z.dtype == expected_z_dtype, z.dtype
assert (x + y).dtype == expected_xy_dtype, (x + y).dtype
print("challenge 1.2: ok")

<details><summary>Solution</summary>

```python
expected_x_dtype = t.int64
expected_y_dtype = t.float32
expected_z_dtype = t.bool
expected_xy_dtype = t.float32
```
</details>

In [ ]:
# Challenge 1.3 — `stack_pairs(origin, directions)`:
#   origin: shape (3,)          directions: shape (n, 3)
#   returns shape (n, 2, 3) where out[i, 0] == origin and out[i, 1] == directions[i]
# Do it WITHOUT a Python loop. (Hints: t.stack, or broadcast + expand.)
def stack_pairs(origin: Tensor, directions: Tensor) -> Tensor:
    raise NotImplementedError()

o = t.tensor([0.0, 0.0, 0.0])
d = t.tensor([[1.0, -1.0, 0.0], [1.0, 0.0, 0.0], [1.0, 1.0, 0.0]])
got = stack_pairs(o, d)
check(got, t.tensor([[[0,0,0],[1,-1,0]], [[0,0,0],[1,0,0]], [[0,0,0],[1,1,0]]], dtype=t.float32), "challenge 1.3")

<details><summary>Solution</summary>

```python
def stack_pairs(origin: Tensor, directions: Tensor) -> Tensor:
    origins = origin.expand_as(directions)          # (n, 3), no copy
    return t.stack([origins, directions], dim=1)    # (n, 2, 3)
```
</details>

# 2️⃣ Evenly-spaced values: `linspace` vs `arange`

The y-coordinates of the directions in `make_rays_1d(9, 1.0)` are `-1, -0.75, -0.5, ..., 0.75, 1` — nine values, both endpoints included. That is exactly `t.linspace(-1, 1, 9)`.

The two tools for evenly-spaced values differ in what you specify:

| | you give | endpoint | count |
|---|---|---|---|
| `t.linspace(start, end, steps)` | start, end, **how many** | **inclusive** | exactly `steps` |
| `t.arange(start, end, step)` | start, end, **step size** | **exclusive** | `ceil((end-start)/step)` |

## Worked examples

In [ ]:
# Example 2.1 — linspace: 5 values from -1 to 1 inclusive → spacing is 2/(5-1) = 0.5
print(t.linspace(-1, 1, 5))       # tensor([-1.0, -0.5, 0.0, 0.5, 1.0])
print(t.linspace(0, 10, 3))       # tensor([0., 5., 10.])
print(t.linspace(0, 10, 1))       # a single value: the start

In [ ]:
# Example 2.2 — arange: step size, stop excluded
print(t.arange(0, 5))             # tensor([0, 1, 2, 3, 4])  int64 because all args are ints
print(t.arange(0, 5, 2))          # tensor([0, 2, 4])
print(t.arange(0.0, 1.0, 0.25))   # tensor([0.00, 0.25, 0.50, 0.75])  — 1.0 is NOT included

In [ ]:
# Example 2.3 — the classic trap: to get an inclusive end with arange you need to overshoot,
# and float steps can give you an extra/missing element from rounding. linspace is safer for "n values between a and b".
print(t.arange(-1, 1.0001, 0.5))  # works, but fragile
print(t.linspace(-1, 1, 5))       # same values, robust

In [ ]:
# Example 2.4 — spacing formula. For linspace(a, b, n), consecutive gap = (b - a) / (n - 1)
v = t.linspace(-10, 10, 9)
print(v)
print(v[1] - v[0])                # 2.5 == 20 / 8

## Faded problems

In [ ]:
# Faded 2.1 — 7 values from -3 to 3 inclusive
ys = t.linspace(..., ..., ...)
check(ys, t.tensor([-3.0, -2.0, -1.0, 0.0, 1.0, 2.0, 3.0]), "faded 2.1")

<details><summary>Solution</summary>

```python
ys = t.linspace(-3, 3, 7)
```
</details>

In [ ]:
# Faded 2.2 — the integers 2, 4, 6, 8 using arange
evens = t.arange(..., ..., ...)
assert evens.tolist() == [2, 4, 6, 8], evens
print("faded 2.2: ok")

<details><summary>Solution</summary>

```python
evens = t.arange(2, 10, 2)  # or t.arange(2, 9, 2)
```
</details>

In [ ]:
# Faded 2.3 — 9 values from -y_limit to +y_limit, as in make_rays_1d(9, 10.0)
y_limit = 10.0
ys = t.linspace(-..., ..., 9)
check(ys, t.tensor([-10.0, -7.5, -5.0, -2.5, 0.0, 2.5, 5.0, 7.5, 10.0]), "faded 2.3")

<details><summary>Solution</summary>

```python
ys = t.linspace(-y_limit, y_limit, 9)
```
</details>

In [ ]:
# Faded 2.4 — how many elements? Fill the literal integers you predict.
n_a = ...   # len(t.linspace(0, 1, 50))
n_b = ...   # len(t.arange(0, 1, 0.1))
n_c = ...   # len(t.arange(0, 10, 3))
assert n_a == len(t.linspace(0, 1, 50))
assert n_b == len(t.arange(0, 1, 0.1))
assert n_c == len(t.arange(0, 10, 3))
print("faded 2.4: ok")

<details><summary>Solution</summary>

```python
n_a = 50
n_b = 10   # 0.0 ... 0.9
n_c = 4    # 0, 3, 6, 9
```
</details>

## Challenge problems

In [ ]:
# Challenge 2.1 — `symmetric(limit, n)`: n values from -limit to +limit inclusive (assume n >= 2).
# Must satisfy: first == -limit, last == +limit, and the sequence is its own negation reversed.
def symmetric(limit: float, n: int) -> Tensor:
    raise NotImplementedError()

s = symmetric(4.0, 5)
check(s, t.tensor([-4.0, -2.0, 0.0, 2.0, 4.0]), "challenge 2.1 values")
assert t.allclose(symmetric(1.5, 8), -symmetric(1.5, 8).flip(0)), "challenge 2.1 symmetry"

<details><summary>Solution</summary>

```python
def symmetric(limit: float, n: int) -> Tensor:
    return t.linspace(-limit, limit, n)
```
</details>

In [ ]:
# Challenge 2.2 — reproduce t.linspace(a, b, n) using ONLY t.arange and arithmetic (no linspace).
# Careful with n == 1.
def my_linspace(a: float, b: float, n: int) -> Tensor:
    raise NotImplementedError()

for (a, b, n) in [(-1, 1, 9), (0, 10, 3), (2.5, 2.5, 4), (0, 1, 1)]:
    check(my_linspace(a, b, n), t.linspace(a, b, n), f"challenge 2.2 {(a,b,n)}")

<details><summary>Solution</summary>

```python
def my_linspace(a: float, b: float, n: int) -> Tensor:
    if n == 1:
        return t.tensor([float(a)])
    step = (b - a) / (n - 1)
    return a + step * t.arange(n, dtype=t.float32)
```
</details>

In [ ]:
# Challenge 2.3 — pixel CENTRES rather than pixel EDGES.
# A screen from -limit to +limit is split into n equal pixels. Return the centre of each pixel.
# e.g. centres(1.0, 4) → [-0.75, -0.25, 0.25, 0.75]
# (Two valid routes: linspace on the edges then average neighbours, or linspace with a half-pixel inset.)
def centres(limit: float, n: int) -> Tensor:
    raise NotImplementedError()

check(centres(1.0, 4), t.tensor([-0.75, -0.25, 0.25, 0.75]), "challenge 2.3 a")
check(centres(2.0, 1), t.tensor([0.0]), "challenge 2.3 b")
check(centres(3.0, 3), t.tensor([-2.0, 0.0, 2.0]), "challenge 2.3 c")

<details><summary>Solution</summary>

```python
def centres(limit: float, n: int) -> Tensor:
    edges = t.linspace(-limit, limit, n + 1)
    return (edges[:-1] + edges[1:]) / 2
    # alternative: half = limit / n; return t.linspace(-limit + half, limit - half, n)
```
</details>

# 3️⃣ Writing into slices: views, broadcasting on assignment, `out=`

The reference solution never builds the direction vectors as separate objects. It allocates zeros, then writes *into* two slices:

```python
rays[:, 1, 0] = 1                                   # every ray's direction x = 1
t.linspace(-y_limit, y_limit, num_pixels, out=rays[:, 1, 1])   # direction y
```

Three ideas make this work:

* **Basic indexing returns a view.** `rays[:, 1, 1]` is not a copy; it shares memory with `rays`. Writing to it writes to `rays`.
* **Broadcasting on assignment.** A scalar (or a smaller tensor) on the right is stretched to the slice's shape.
* **`out=`.** Many torch functions accept `out=some_view` and write their result straight into it.

## Worked examples

In [ ]:
# Example 3.1 — a slice is a view; mutating it mutates the original
m = t.zeros((3, 2, 3))
col = m[:, 1, 0]                  # shape (3,): the "direction x" entry of each of the 3 rays
col[:] = 1                        # in-place write through the view
print(m[:, 1, :])                 # x-column is now 1 for all rows
print(col.untyped_storage().data_ptr() == m.untyped_storage().data_ptr())   # True: same underlying storage

In [ ]:
# Example 3.2 — broadcasting on assignment: scalar, then a 1-D tensor
m = t.zeros((4, 2, 3))
m[:, 1, 0] = 1                    # scalar → all 4 entries
m[:, 1, 1] = t.tensor([-1.0, -0.5, 0.5, 1.0])   # shape (4,) matches slice shape (4,)
print(m[:, 1, :])

In [ ]:
# Example 3.3 — out= writes the result of a function directly into an existing view
m = t.zeros((5, 2, 3))
t.linspace(-2, 2, 5, out=m[:, 1, 1])
print(m[:, 1, 1])                 # tensor([-2., -1., 0., 1., 2.])

In [ ]:
# Example 3.4 — shape mismatch on assignment is an error (broadcasting only stretches size-1 axes)
m = t.zeros((4, 2, 3))
try:
    m[:, 1, 1] = t.tensor([1.0, 2.0, 3.0])   # (3,) into a (4,) slice
except RuntimeError as e:
    print("RuntimeError:", str(e)[:80], "...")

In [ ]:
# Example 3.5 — advanced (fancy) indexing returns a COPY, not a view. Writing to it does nothing to the original.
m = t.zeros(5)
sub = m[[0, 2, 4]]                # index with a list → copy
sub[:] = 9
print(m)                          # still zeros!
m[[0, 2, 4]] = 9                  # ...but direct assignment through fancy indexing DOES work
print(m)

## Faded problems

In [ ]:
# Faded 3.1 — set the x-component of every direction to 1
rays = t.zeros((6, 2, 3))
rays[..., ..., ...] = 1
check(rays[:, 1, 0], t.ones(6), "faded 3.1 x")
check(rays[:, 0, :], t.zeros((6, 3)), "faded 3.1 origin untouched")

<details><summary>Solution</summary>

```python
rays[:, 1, 0] = 1
```
</details>

In [ ]:
# Faded 3.2 — write linspace(-1, 1, 6) into the y-component of every direction using out=
rays = t.zeros((6, 2, 3))
t.linspace(..., ..., ..., out=rays[..., ..., ...])
check(rays[:, 1, 1], t.linspace(-1, 1, 6), "faded 3.2")
assert rays[:, 1, 0].sum() == 0 and rays[:, 1, 2].sum() == 0, "faded 3.2 wrote to the wrong column"

<details><summary>Solution</summary>

```python
t.linspace(-1, 1, 6, out=rays[:, 1, 1])
```
</details>

In [ ]:
# Faded 3.3 — same result as 3.2, but WITHOUT out=: plain slice assignment
rays = t.zeros((6, 2, 3))
rays[:, 1, 1] = ...
check(rays[:, 1, 1], t.linspace(-1, 1, 6), "faded 3.3")

<details><summary>Solution</summary>

```python
rays[:, 1, 1] = t.linspace(-1, 1, 6)
```
</details>

In [ ]:
# Faded 3.4 — assign a whole direction vector at once to ray number 2 (0-indexed)
rays = t.zeros((4, 2, 3))
rays[..., ...] = t.tensor([1.0, 0.25, 0.0])
check(rays[2, 1], t.tensor([1.0, 0.25, 0.0]), "faded 3.4 row 2")
assert rays.sum() == 1.25, "faded 3.4 wrote elsewhere"

<details><summary>Solution</summary>

```python
rays[2, 1] = t.tensor([1.0, 0.25, 0.0])
```
</details>

## Challenge problems

In [ ]:
# Challenge 3.1 — is it a view? For each expression on `base`, predict True (view: shares storage) or False (copy).
base = t.arange(12.0).reshape(3, 4)
pred_a = ...   # base[1]
pred_b = ...   # base[:, 2]
pred_c = ...   # base[[0, 2]]
pred_d = ...   # base.reshape(12)
pred_e = ...   # base[base > 5]
pred_f = ...   # base.T

def shares(x): return x.untyped_storage().data_ptr() == base.untyped_storage().data_ptr()
assert pred_a == shares(base[1])
assert pred_b == shares(base[:, 2])
assert pred_c == shares(base[[0, 2]])
assert pred_d == shares(base.reshape(12))
assert pred_e == shares(base[base > 5])
assert pred_f == shares(base.T)
print("challenge 3.1: ok")

<details><summary>Solution</summary>

```python
pred_a = True    # basic indexing
pred_b = True    # basic slicing
pred_c = False   # fancy (list) indexing copies
pred_d = True    # reshape of a contiguous tensor is a view
pred_e = False   # boolean mask indexing copies
pred_f = True    # transpose is a view (strides change, storage doesn't)
```
</details>

In [ ]:
# Challenge 3.2 — `fill_directions(rays, xs, ys)` mutates `rays` IN PLACE (returns None):
#   rays: (n, 2, 3) zeros;  xs: (n,) ;  ys: (n,)
#   after the call, rays[i, 1] == (xs[i], ys[i], 0) and rays[i, 0] is untouched.
# No loops, no new (n, 2, 3) tensor. Then the assertion checks you really mutated the original object.
def fill_directions(rays: Tensor, xs: Tensor, ys: Tensor) -> None:
    raise NotImplementedError()

r = t.zeros((3, 2, 3))
r_id = id(r)
fill_directions(r, t.tensor([1.0, 1.0, 1.0]), t.tensor([-1.0, 0.0, 1.0]))
assert id(r) == r_id
check(r[:, 1], t.tensor([[1.0, -1.0, 0.0], [1.0, 0.0, 0.0], [1.0, 1.0, 0.0]]), "challenge 3.2 dirs")
check(r[:, 0], t.zeros((3, 3)), "challenge 3.2 origins")

<details><summary>Solution</summary>

```python
def fill_directions(rays: Tensor, xs: Tensor, ys: Tensor) -> None:
    rays[:, 1, 0] = xs
    rays[:, 1, 1] = ys
```
</details>

In [ ]:
# Challenge 3.3 — broadcasting on assignment with a 2-D right-hand side.
# Set ALL origins (rays[:, 0, :]) of an (n, 2, 3) tensor to the vector `origin`, shape (3,), in ONE assignment.
# Then set ALL directions to `direction`, shape (3,), in ONE assignment.
def set_all(rays: Tensor, origin: Tensor, direction: Tensor) -> None:
    raise NotImplementedError()

r = t.zeros((4, 2, 3))
set_all(r, t.tensor([0.0, 0.0, 5.0]), t.tensor([1.0, 2.0, 3.0]))
check(r[:, 0], t.tensor([0.0, 0.0, 5.0]).expand(4, 3), "challenge 3.3 origins")
check(r[:, 1], t.tensor([1.0, 2.0, 3.0]).expand(4, 3), "challenge 3.3 directions")

<details><summary>Solution</summary>

```python
def set_all(rays: Tensor, origin: Tensor, direction: Tensor) -> None:
    rays[:, 0] = origin       # (3,) broadcasts over the (n, 3) slice
    rays[:, 1] = direction
```
</details>

# 4️⃣ Rays as `(origin, direction)` pairs

A **ray** is a half-line: it starts at an origin $O$ and heads in a direction $D$. Every point on it is

$$P(u) = O + u\,D, \qquad u \ge 0.$$

In this chapter a ray is stored as a `(2, 3)` tensor: row 0 is $O$, row 1 is $D$, and each row is `(x, y, z)`. A batch of $n$ rays is `(n, 2, 3)`.

Two facts matter for `make_rays_1d`:

* The camera is at the origin `(0, 0, 0)` and the screen is the plane `x = 1`. A ray from the camera through screen point `(1, y, 0)` therefore has direction `(1, y, 0)` — with $u = 1$ the ray is *on* the screen.
* The $z$ coordinate stays 0: we're working in the $xy$-plane for now, but keep 3 dims so the tensors line up with the 3D code later.

## Worked examples

In [ ]:
# Example 4.1 — walking along a ray: P(u) = O + u D
O = t.tensor([0.0, 0.0, 0.0])
D = t.tensor([1.0, -0.5, 0.0])
for u in [0.0, 1.0, 2.0]:
    print(u, O + u * D)         # u=1 lands on the screen x=1 at y=-0.5

In [ ]:
# Example 4.2 — a ray as a (2, 3) tensor, and reading its parts back
ray = t.tensor([[0.0, 0.0, 0.0], [1.0, 0.75, 0.0]])
origin, direction = ray          # unpacking along dim 0
print("origin   ", origin)
print("direction", direction)
print("dir y    ", ray[1, 1])    # [row=direction, col=y]

In [ ]:
# Example 4.3 — a batch of rays: (n, 2, 3). Index order is [ray, point, coord].
rays = t.tensor([
    [[0.0, 0.0, 0.0], [1.0, -1.0, 0.0]],
    [[0.0, 0.0, 0.0], [1.0,  0.0, 0.0]],
    [[0.0, 0.0, 0.0], [1.0,  1.0, 0.0]],
])
print(rays.shape)
print(rays[:, 1, 1])              # all direction-y values: tensor([-1., 0., 1.])
print(rays[2])                    # the whole third ray

In [ ]:
# Example 4.4 — where does each ray hit the screen x = 1?  Solve O_x + u D_x = 1 for u, then P(u).
O = rays[:, 0]                    # (n, 3)
D = rays[:, 1]                    # (n, 3)
u = (1 - O[:, 0]) / D[:, 0]       # (n,)
hits = O + u[:, None] * D         # broadcast u over the coord axis
print(hits)                       # every x is 1; y matches direction y because O = 0 and D_x = 1

## Faded problems

In [ ]:
# Faded 4.1 — the point at parameter u along the ray
def point_at(ray: Tensor, u: float) -> Tensor:
    origin, direction = ray
    return ... + ... * ...

r = t.tensor([[0.0, 1.0, 0.0], [2.0, 0.0, 0.0]])
check(point_at(r, 0.0), t.tensor([0.0, 1.0, 0.0]), "faded 4.1 u=0")
check(point_at(r, 1.5), t.tensor([3.0, 1.0, 0.0]), "faded 4.1 u=1.5")

<details><summary>Solution</summary>

```python
def point_at(ray: Tensor, u: float) -> Tensor:
    origin, direction = ray
    return origin + u * direction
```
</details>

In [ ]:
# Faded 4.2 — build ONE ray from the camera at (0,0,0) through screen point (1, y, 0)
def ray_through(y: float) -> Tensor:
    return t.tensor([[0.0, 0.0, 0.0], [..., ..., ...]])

check(ray_through(-0.25), t.tensor([[0.0, 0.0, 0.0], [1.0, -0.25, 0.0]]), "faded 4.2")

<details><summary>Solution</summary>

```python
def ray_through(y: float) -> Tensor:
    return t.tensor([[0.0, 0.0, 0.0], [1.0, y, 0.0]])
```
</details>

In [ ]:
# Faded 4.3 — extract the direction-y of every ray in a batch, and the origin of every ray
rays = t.zeros((5, 2, 3)); rays[:, 1, 0] = 1; rays[:, 1, 1] = t.linspace(-2, 2, 5)
dir_y = rays[..., ..., ...]
origins = rays[..., ...]
check(dir_y, t.linspace(-2, 2, 5), "faded 4.3 dir_y")
check(origins, t.zeros((5, 3)), "faded 4.3 origins")

<details><summary>Solution</summary>

```python
dir_y = rays[:, 1, 1]
origins = rays[:, 0]
```
</details>

In [ ]:
# Faded 4.4 — screen hit points for a batch, assuming origin = 0 and direction x = 1 (so u = 1)
def screen_hits(rays: Tensor) -> Tensor:
    origins = rays[:, 0]
    directions = rays[:, 1]
    u = ...
    return origins + u * directions

check(screen_hits(rays), t.stack([t.ones(5), t.linspace(-2, 2, 5), t.zeros(5)], dim=1), "faded 4.4")

<details><summary>Solution</summary>

```python
def screen_hits(rays: Tensor) -> Tensor:
    origins = rays[:, 0]
    directions = rays[:, 1]
    u = 1.0
    return origins + u * directions
```
</details>

## Challenge problems

In [ ]:
# Challenge 4.1 — `screen_hits_general(rays, screen_x)`: rays (n, 2, 3) with ANY origin and ANY nonzero
# direction x. Return the (n, 3) points where each ray meets the plane x = screen_x. No loops.
# A ray is a HALF-line (u >= 0): if the plane is behind the origin, return NaN for that ray's row.
# (Hint: compute u, then `t.where(u[:, None] >= 0, hits, t.nan)`.)
def screen_hits_general(rays: Tensor, screen_x: float) -> Tensor:
    raise NotImplementedError()

rs = t.tensor([
    [[0.0, 0.0, 0.0], [1.0, 2.0, 0.0]],      # hits x=3 at u=3 → (3, 6, 0)
    [[1.0, 1.0, 1.0], [2.0, 0.0, -1.0]],     # hits x=3 at u=1 → (3, 1, 0)
    [[-1.0, 0.0, 0.0], [0.5, 0.5, 0.5]],     # hits x=3 at u=8 → (3, 4, 4)
])
check(screen_hits_general(rs, 3.0), t.tensor([[3.0, 6.0, 0.0], [3.0, 1.0, 0.0], [3.0, 4.0, 4.0]]), "challenge 4.1")
behind = screen_hits_general(t.tensor([[[2.0, 0.0, 0.0], [1.0, 1.0, 0.0]]]), 1.0)   # plane x=1 is behind origin x=2
assert behind.shape == (1, 3) and behind.isnan().all(), "challenge 4.1: a plane behind the origin must give NaN"

<details><summary>Solution</summary>

```python
def screen_hits_general(rays: Tensor, screen_x: float) -> Tensor:
    O, D = rays[:, 0], rays[:, 1]
    u = (screen_x - O[:, 0]) / D[:, 0]        # (n,)
    hits = O + u[:, None] * D
    return t.where(u[:, None] >= 0, hits, t.nan)   # half-line: nothing behind the origin
```
</details>

In [ ]:
# Challenge 4.2 — same ray, different tensor. Two (origin, direction) pairs describe the SAME half-line
# iff origins are equal and directions are positive multiples of each other. Write `same_ray(a, b) -> bool`.
def same_ray(a: Tensor, b: Tensor) -> bool:
    raise NotImplementedError()

A = t.tensor([[0.0, 0.0, 0.0], [1.0, 0.5, 0.0]])
assert same_ray(A, t.tensor([[0.0, 0.0, 0.0], [2.0, 1.0, 0.0]])) is True      # scaled by 2
assert same_ray(A, t.tensor([[0.0, 0.0, 0.0], [-1.0, -0.5, 0.0]])) is False   # opposite direction
assert same_ray(A, t.tensor([[0.0, 1.0, 0.0], [1.0, 0.5, 0.0]])) is False     # different origin
assert same_ray(A, t.tensor([[0.0, 0.0, 0.0], [1.0, 0.6, 0.0]])) is False     # not parallel
tiny = t.tensor([[0.0, 0.0, 0.0], [1e-4, 0.0, 0.0]])
assert same_ray(tiny, t.tensor([[0.0, 0.0, 0.0], [1e-4, 1e-4, 0.0]])) is False  # 45° apart — tolerance must not depend on scale
assert same_ray(tiny, t.tensor([[0.0, 0.0, 0.0], [5.0, 0.0, 0.0]])) is True      # same direction, wildly different length
print("challenge 4.2: ok")

<details><summary>Solution</summary>

```python
def same_ray(a: Tensor, b: Tensor) -> bool:
    if not t.allclose(a[0], b[0]):
        return False
    # compare UNIT directions so the tolerance is scale-free
    da, db = a[1] / a[1].norm(), b[1] / b[1].norm()
    return t.allclose(da, db, atol=1e-6)   # equal unit vectors ⇔ parallel AND same sense
```
</details>

In [ ]:
# Challenge 4.3 — pen-and-paper, then code. A camera sits at (0, 0, 0). The screen is x = 1 and spans
# y ∈ [-y_limit, y_limit]. A ray through screen point (1, y, 0) continues to x = 4.
# (a) What is its y-coordinate at x = 4, as a formula in y?   → fill `y_at_4(y)`.
# (b) A segment sits at x = 4 covering y ∈ [-2, 2]. For which y on the screen does the ray hit it?
#     Return the (inclusive) interval as a tuple (lo, hi)  → fill `screen_window()`.
def y_at_4(y: float) -> float:
    raise NotImplementedError()

def screen_window() -> tuple[float, float]:
    raise NotImplementedError()

assert abs(y_at_4(0.25) - 1.0) < 1e-9
assert abs(y_at_4(-1.0) + 4.0) < 1e-9
assert screen_window() == (-0.5, 0.5)
print("challenge 4.3: ok")

<details><summary>Solution</summary>

```python
def y_at_4(y: float) -> float:
    return 4 * y          # P(u) = u * (1, y, 0); x = 4 ⇒ u = 4 ⇒ y-coord = 4y

def screen_window() -> tuple[float, float]:
    return (-0.5, 0.5)    # need 4y ∈ [-2, 2]
```
</details>

# 5️⃣ Problems at the level of `make_rays_1d`

Each of these is a small variation on the original. Same difficulty (🔴🔴⚪⚪⚪), same 10–15 minute budget. Aim for the same three-line shape: allocate zeros → write the constant column → write the `linspace` column. Avoid Python loops.

Reference, for comparison (don't peek until you've done at least one):

```python
def make_rays_1d(num_pixels, y_limit):
    rays = t.zeros((num_pixels, 2, 3), dtype=t.float32)
    t.linspace(-y_limit, y_limit, num_pixels, out=rays[:, 1, 1])
    rays[:, 1, 0] = 1
    return rays
```

In [ ]:
# Problem 5.1 — make_rays_1d_z: identical to make_rays_1d but the rays fan out in the z direction
# instead of y. Directions run from (1, 0, -z_limit) to (1, 0, +z_limit) inclusive.
def make_rays_1d_z(num_pixels: int, z_limit: float) -> Tensor:
    raise NotImplementedError()

r = make_rays_1d_z(5, 2.0)
assert r.shape == (5, 2, 3) and r.dtype == t.float32
check(r[:, 0], t.zeros((5, 3)), "5.1 origins")
check(r[:, 1, 0], t.ones(5), "5.1 dir x")
check(r[:, 1, 1], t.zeros(5), "5.1 dir y")
check(r[:, 1, 2], t.tensor([-2.0, -1.0, 0.0, 1.0, 2.0]), "5.1 dir z")

<details><summary>Solution</summary>

```python
def make_rays_1d_z(num_pixels: int, z_limit: float) -> Tensor:
    rays = t.zeros((num_pixels, 2, 3), dtype=t.float32)
    rays[:, 1, 0] = 1
    t.linspace(-z_limit, z_limit, num_pixels, out=rays[:, 1, 2])
    return rays
```
</details>

In [ ]:
# Problem 5.2 — make_rays_1d_from: the camera is at an arbitrary `origin` (shape (3,)) and the screen is the
# plane x = origin_x + 1. Rays still fan out in y from -y_limit to +y_limit RELATIVE to the origin.
# So direction is (1, y, 0) exactly as before, but every origin row equals `origin`.
def make_rays_1d_from(num_pixels: int, y_limit: float, origin: Tensor) -> Tensor:
    raise NotImplementedError()

o = t.tensor([-3.0, 1.0, 0.5])
r = make_rays_1d_from(4, 1.5, o)
assert r.shape == (4, 2, 3)
check(r[:, 0], o.expand(4, 3), "5.2 origins")
check(r[:, 1], t.tensor([[1.0, -1.5, 0.0], [1.0, -0.5, 0.0], [1.0, 0.5, 0.0], [1.0, 1.5, 0.0]]), "5.2 directions")

<details><summary>Solution</summary>

```python
def make_rays_1d_from(num_pixels: int, y_limit: float, origin: Tensor) -> Tensor:
    rays = t.zeros((num_pixels, 2, 3), dtype=t.float32)
    rays[:, 0] = origin
    rays[:, 1, 0] = 1
    rays[:, 1, 1] = t.linspace(-y_limit, y_limit, num_pixels)
    return rays
```
</details>

In [ ]:
# Problem 5.3 — make_rays_1d_screen: the screen is at x = screen_x instead of x = 1.
# Direction vectors must point AT the screen point, i.e. direction = (screen_x, y, 0) with y in
# [-y_limit, y_limit] inclusive — so u = 1 still lands on the screen.
def make_rays_1d_screen(num_pixels: int, y_limit: float, screen_x: float) -> Tensor:
    raise NotImplementedError()

r = make_rays_1d_screen(3, 1.0, 2.5)
check(r[:, 0], t.zeros((3, 3)), "5.3 origins")
check(r[:, 1], t.tensor([[2.5, -1.0, 0.0], [2.5, 0.0, 0.0], [2.5, 1.0, 0.0]]), "5.3 directions")
# Sanity: with u = 1 every ray sits on the plane x = screen_x
assert t.allclose((r[:, 0] + r[:, 1])[:, 0], t.full((3,), 2.5))

<details><summary>Solution</summary>

```python
def make_rays_1d_screen(num_pixels: int, y_limit: float, screen_x: float) -> Tensor:
    rays = t.zeros((num_pixels, 2, 3), dtype=t.float32)
    rays[:, 1, 0] = screen_x
    t.linspace(-y_limit, y_limit, num_pixels, out=rays[:, 1, 1])
    return rays
```
</details>

In [ ]:
# Problem 5.4 — make_rays_1d_unit: same fan as make_rays_1d, but every direction is normalised to
# unit length. (Hint: compute directions first, then divide by their norm along the last axis with keepdim.)
def make_rays_1d_unit(num_pixels: int, y_limit: float) -> Tensor:
    raise NotImplementedError()

r = make_rays_1d_unit(3, 1.0)
check(r[:, 0], t.zeros((3, 3)), "5.4 origins")
check(r[:, 1].norm(dim=-1), t.ones(3), "5.4 unit length")
s = 2 ** -0.5
check(r[:, 1], t.tensor([[s, -s, 0.0], [1.0, 0.0, 0.0], [s, s, 0.0]]), "5.4 directions")

<details><summary>Solution</summary>

```python
def make_rays_1d_unit(num_pixels: int, y_limit: float) -> Tensor:
    rays = t.zeros((num_pixels, 2, 3), dtype=t.float32)
    rays[:, 1, 0] = 1
    rays[:, 1, 1] = t.linspace(-y_limit, y_limit, num_pixels)
    rays[:, 1] /= rays[:, 1].norm(dim=-1, keepdim=True)
    return rays
```
</details>

In [ ]:
# Problem 5.5 — make_rays_1d_centres: like make_rays_1d, but the screen [-y_limit, y_limit] is divided into
# num_pixels equal pixels and each ray goes through the CENTRE of its pixel (see Challenge 2.3).
# make_rays_1d_centres(4, 1.0) has direction-y values [-0.75, -0.25, 0.25, 0.75].
def make_rays_1d_centres(num_pixels: int, y_limit: float) -> Tensor:
    raise NotImplementedError()

r = make_rays_1d_centres(4, 1.0)
check(r[:, 1, 0], t.ones(4), "5.5 dir x")
check(r[:, 1, 1], t.tensor([-0.75, -0.25, 0.25, 0.75]), "5.5 dir y")
check(make_rays_1d_centres(1, 3.0)[:, 1, 1], t.tensor([0.0]), "5.5 single pixel")

<details><summary>Solution</summary>

```python
def make_rays_1d_centres(num_pixels: int, y_limit: float) -> Tensor:
    rays = t.zeros((num_pixels, 2, 3), dtype=t.float32)
    rays[:, 1, 0] = 1
    half = y_limit / num_pixels
    rays[:, 1, 1] = t.linspace(-y_limit + half, y_limit - half, num_pixels)
    return rays
```
</details>

In [ ]:
# Problem 5.6 — make_rays_1d_range: rays whose screen y runs from y_min to y_max (not symmetric).
# make_rays_1d_range(3, 0.0, 1.0) has direction-y [0.0, 0.5, 1.0].
# Then confirm that make_rays_1d_range(n, -L, L) matches the reference make_rays_1d(n, L) exactly.
def make_rays_1d_range(num_pixels: int, y_min: float, y_max: float) -> Tensor:
    raise NotImplementedError()

def make_rays_1d(num_pixels, y_limit):   # reference, for the comparison
    rays = t.zeros((num_pixels, 2, 3), dtype=t.float32)
    t.linspace(-y_limit, y_limit, num_pixels, out=rays[:, 1, 1])
    rays[:, 1, 0] = 1
    return rays

check(make_rays_1d_range(3, 0.0, 1.0)[:, 1, 1], t.tensor([0.0, 0.5, 1.0]), "5.6 asymmetric")
check(make_rays_1d_range(9, -10.0, 10.0), make_rays_1d(9, 10.0), "5.6 matches reference")

<details><summary>Solution</summary>

```python
def make_rays_1d_range(num_pixels: int, y_min: float, y_max: float) -> Tensor:
    rays = t.zeros((num_pixels, 2, 3), dtype=t.float32)
    rays[:, 1, 0] = 1
    t.linspace(y_min, y_max, num_pixels, out=rays[:, 1, 1])
    return rays
```
</details>

## Where next

If every cell above passes, go back to `0.1_Ray_Tracing_exercises.ipynb` and do `make_rays_1d` from scratch without looking at the reference. Then `render_lines_with_plotly(make_rays_1d(9, 10.0))` should show a fan of nine rays from the origin.

The next exercise, `intersect_ray_1d`, leans on Section 4 (the $O + uD$ parametrisation) plus solving a $2\times2$ linear system with `t.linalg.solve`.